# About this Notebook

Automated Finance Research Report with LangGraph + OpenAI + Exa + yfinance

This example extends the earlier pipeline:

1) GatherData: fetches ticker data via yfinance and fresh headlines via Exa
2) ExtractSignals: LLM turns raw data into structured bullets
3) DraftReport: LLM drafts a concise markdown research note
4) Review: LLM critiques the draft
5) Improve: optional revision loop

__Setup:__
- set OPENAI_API_KEY and EXA_API_KEY in your environment


# Dependencies

In [1]:
!pip install -q langchain==0.3.27 \
                langgraph==0.6.7 \
                langchain-openai==0.3.32 \
                langchain_experimental==0.3.4 \
                langchain_community==0.3.29 \
                python-dotenv==1.0.1 \
                langchain-core==0.3.75 \
                serpapi==0.1.5 \
                google-search-results==2.4.2 \
                exa_py==1.7.0

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confli

# Imports for API

In [2]:

from dotenv import load_dotenv
import os

# Imports

In [ ]:
from __future__ import annotations


import os
import json
from typing import TypedDict, List, Dict, Any
from datetime import datetime, timedelta

import yfinance as yf
from exa_py import Exa
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END


# Setup Env

In [ ]:
load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0.1)
exa = Exa(os.getenv("EXA_API_KEY"))


In [ ]:
# Setup Graphstate

In [ ]:
class ResearchState(TypedDict):
    ticker: str
    date: str
    price_data: List[Dict[str, Any]]
    headlines: List[str]
    bullets: Dict[str, Any]
    draft_md: str
    critique: Dict[str, Any]
    report_md: str
    iter: int


# Structured Models

In [ ]:
class ExtractedSignals(BaseModel):
    trend_summary: str
    drivers: List[str]
    risks: List[str]
    data_highlights: List[str]

class Critique(BaseModel):
    needs_revision: bool
    strengths: List[str] = []
    suggestions: List[str] = []
    risk_flags: List[str] = []


# Helpers

In [ ]:
def gather_data(state: ResearchState) -> ResearchState:
    ticker = state["ticker"]
    end = datetime.today()
    start = end - timedelta(days=7)
    df = yf.download(ticker, start=start, end=end)
    price_data = [
        {"date": idx.strftime("%Y-%m-%d"), "close": float(row["Close"]), "volume": int(row["Volume"])}
        for idx, row in df.iterrows()
    ]

    # use Exa to fetch news for last 7 days
    date_cutoff = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
    search_response = exa.search_and_contents(
        ticker, use_autoprompt=True, start_published_date=date_cutoff
    )
    headlines = [r.title for r in search_response.results[:5] if r.title]

    return {**state, "price_data": price_data, "headlines": headlines}


def extract_signals(state: ResearchState) -> ResearchState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an equity research associate. Extract compact signals from the data."),
        ("human", "Ticker: {ticker}\nDate: {date}\nPrices:```json\n{prices}\n```\nHeadlines:```\n{headlines}\n```\nReturn JSON."),
    ])
    chain = prompt | llm.with_structured_output(ExtractedSignals)
    res = chain.invoke({
        "ticker": state["ticker"],
        "date": state["date"],
        "prices": json.dumps(state["price_data"], indent=2),
        "headlines": "\n".join(state["headlines"]),
    })
    return {**state, "bullets": res.model_dump()}


def draft_report(state: ResearchState) -> ResearchState:
    sys = SystemMessage(content=(
        "You write concise sell‑side style research notes. Sections: Summary, Drivers, Technicals, Risks, Outlook, Sources."
    ))
    human = HumanMessage(content=(
        f"Ticker: {state['ticker']}\nDate: {state['date']}\n"
        f"Signals:```json\n{json.dumps(state['bullets'], indent=2)}\n```\n"
        "Draft in Markdown."
    ))
    text = (llm | StrOutputParser()).invoke([sys, human])
    return {**state, "draft_md": text, "report_md": text}


def review(state: ResearchState) -> ResearchState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a meticulous equity research reviewer."),
        ("human", "Review this note and return JSON critique.\nNote:\n```\n{note}\n```"),
    ])
    chain = prompt | llm.with_structured_output(Critique)
    res = chain.invoke({"note": state["report_md"]})
    return {**state, "critique": res.model_dump()}


def improve(state: ResearchState) -> ResearchState:
    if not state.get("critique", {}).get("needs_revision"):
        return state
    sys = SystemMessage(content="Revise the note using reviewer suggestions. Keep it under 450 words.")
    human = HumanMessage(content=(
        "Original note:\n```\n" + state["report_md"] + "\n```\n\n"
        "Suggestions:\n" + json.dumps(state["critique"].get("suggestions", []), indent=2)
    ))
    revised = (llm | StrOutputParser()).invoke([sys, human])
    return {**state, "report_md": revised}

# Graph

In [ ]:
graph = StateGraph(ResearchState)
graph.add_node("gather", gather_data)
graph.add_node("signals", extract_signals)
graph.add_node("draft", draft_report)
graph.add_node("review", review)
graph.add_node("improve", improve)

# wiring
graph.add_edge(START, "gather")
graph.add_edge("gather", "signals")
graph.add_edge("signals", "draft")
graph.add_edge("draft", "review")

def needs_more(state: ResearchState) -> str:
    if state.get("critique", {}).get("needs_revision", False) and state.get("iter", 0) < 1:
        return "improve"
    return "end"

graph.add_conditional_edges("review", needs_more, {"improve": "improve", "end": END})
graph.add_edge("improve", END)

app = graph.compile()

# Run Application

In [7]:
if __name__ == "__main__":
    init: ResearchState = {
        "ticker": "AAPL",
        "date": datetime.today().strftime("%Y-%m-%d"),
        "price_data": [],
        "headlines": [],
        "bullets": {},
        "draft_md": "",
        "critique": {},
        "report_md": "",
        "iter": 0,
    }

    from rich.console import Console
    from rich.panel import Panel

    console = Console()

    for event in app.stream(init):
        for node, output in event.items():
            console.rule(f"[bold]Node: {node}")
            if node == "signals":
                console.print(Panel.fit(json.dumps(output["bullets"], indent=2)))
            elif node in ("draft", "improve"):
                console.print(output["report_md"])
            elif node == "review":
                console.print(Panel.fit(json.dumps(output["critique"], indent=2)))

    final = app.invoke(init)
    console.rule("[bold green]Final Report")
    console.print(final["report_md"])


/tmp/ipython-input-2133453628.py:73: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-2133453628.py:75: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  {"date": idx.strftime("%Y-%m-%d"), "close": float(row["Close"]), "volume": int(row["Volume"])}
/tmp/ipython-input-2133453628.py:75: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  {"date": idx.strftime("%Y-%m-%d"), "close": float(row["Close"]), "volume": int(row["Volume"])}


────────────────────────────────────────────────── Node: gather ───────────────────────────────────────────────────

────────────────────────────────────────────────── Node: signals ──────────────────────────────────────────────────

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "trend_summary": "AAPL has shown a declining trend over the past week, with a notable drop from a peak of     │
│ $239.78 on September 4 to $234.35 on September 9.",                                                             │
│   "drivers": [                                                                                                  │
│     "Recent market volatility affecting tech stocks",                                                           │
│     "Investor sentiment following recent earnings reports",                                                     │
│     "Anticipation of new product launches affecting stock performance"                                          │
│   ],                                                                                                            │
│   "risks": [                                                                                                    │
│     "Potential supply chain disruptions",                                                                       │
│     "Increased competition in the tech sector",                                                                 │
│     "Regulatory challenges impacting operations"                                                                │
│   ],                                                                                                            │
│   "data_highlights": [                                                                                          │
│     "Highest closing price in the week was $239.78 on September 4",                                             │
│     "Lowest closing price in the week was $234.35 on September 9",                                              │
│     "Volume peaked at 66,427,800 on September 3"                                                                │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────────── Node: draft ───────────────────────────────────────────────────

...

────────────────────────────────────────────────── Node: improve ──────────────────────────────────────────────────

# AAPL Research Note  
**Date:** September 10, 2025  

## Summary  
AAPL has seen a decline of approximately 2.98% over the past week, dropping from a peak of $239.78 on September 4 
to $234.35 on September 9. This downturn is primarily due to recent market volatility, mixed investor sentiment 
following earnings reports, and uncertainty surrounding upcoming product launches.

## Drivers  
- **Market Volatility:** The tech sector has experienced fluctuations, with the Nasdaq Composite Index down about 
1.5% over the same period, impacting investor confidence in AAPL.
- **Earnings Reports:** AAPL's recent earnings report showed a 5% year-over-year revenue increase, but earnings per
share (EPS) of $1.20 fell short of analyst expectations of $1.25, leading to mixed reactions among investors.
- **Product Launch Anticipation:** Speculation around new product announcements, including the iPhone 15, has 
created uncertainty, with investors cautious about potential supply chain issues.

## Technicals  
- **Weekly High:** $239.78 (September 4)  
- **Weekly Low:** $234.35 (September 9)  
- **Volume Peak:** 66,427,800 shares (September 3)  
- **Percentage Change:** -2.98% from the weekly high to low  

## Risks  
- **Supply Chain Disruptions:** Ongoing global supply chain challenges could affect production and delivery 
timelines, particularly for new product launches.
- **Increased Competition:** Competitors like Samsung and Google are ramping up their product offerings, which may 
impact AAPL's market share and pricing strategies.
- **Regulatory Challenges:** Increased scrutiny from regulators regarding privacy and antitrust issues could hinder
operational flexibility and growth.

## Outlook  
Considering the current market conditions and identified risks, AAPL may continue to face downward pressure in the 
short term. However, successful execution of upcoming product launches could serve as a catalyst for recovery, 
especially if they resonate well with consumers and analysts alike.

## Historical Context  
Historically, AAPL's stock has shown resilience following product launches, often recovering from short-term 
declines. For instance, after the launch of the iPhone 14, the stock rebounded by over 10% within a month. This 
historical performance suggests that if the upcoming launches meet or exceed expectations, AAPL could see a similar
recovery trajectory.

## Sources  
- Market data from September 2025  
- Recent earnings reports and analyst commentary  
- Industry news regarding product launches and competition

/tmp/ipython-input-2133453628.py:73: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/tmp/ipython-input-2133453628.py:75: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  {"date": idx.strftime("%Y-%m-%d"), "close": float(row["Close"]), "volume": int(row["Volume"])}
/tmp/ipython-input-2133453628.py:75: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  {"date": idx.strftime("%Y-%m-%d"), "close": float(row["Close"]), "volume": int(row["Volume"])}


────────────────────────────────────────────────── Final Report ───────────────────────────────────────────────────

# AAPL Research Note  
**Date:** September 10, 2025  

## Summary  
AAPL has experienced a downward trend over the past week, declining from a peak of $239.78 on September 4 to 
$234.35 on September 9, representing a decrease of approximately 2.9%. This decline is attributed to recent market 
volatility and shifting investor sentiment.

## Drivers  
- **Market Volatility:** The tech sector has been particularly affected by fluctuations in the broader market, 
leading to increased caution among investors.  
- **Investor Sentiment:** A shift towards caution has been observed, with many investors reassessing their 
positions in light of recent economic indicators.  
- **Supply Chain Concerns:** Specific issues, such as delays in semiconductor deliveries and rising costs of raw 
materials, have raised concerns about AAPL's production capabilities and timelines.

## Technicals  
- **Highest Closing Price:** $239.78 on September 4  
- **Lowest Closing Price:** $234.35 on September 9  
- **Volume Peak:** 66,427,800 on September 3  

## Risks  
- **Market Downturn:** Continued declines in the market could further depress AAPL's stock price.  
- **Negative News:** Any adverse news or disappointing earnings reports could exacerbate the current downward 
trend.  
- **Increased Competition:** The tech sector is witnessing heightened competition, which may impact AAPL's growth 
prospects.

## Potential Catalysts  
- **Upcoming Product Launches:** AAPL is expected to unveil new products in the coming weeks, which could generate 
renewed interest and drive stock prices higher.  
- **Earnings Reports:** The upcoming quarterly earnings report may provide insights into AAPL's performance and 
reassure investors about its growth trajectory.  
- **Market Recovery:** A stabilization in the broader market could improve investor sentiment and lead to a rebound
in tech stocks, including AAPL.

## Outlook  
Given the current market conditions and potential risks, AAPL may face continued pressure in the near term. 
However, upcoming product launches and earnings reports could serve as catalysts for a reversal of the current 
trend. Investors should closely monitor market sentiment and developments related to supply chain issues.

## Sources  
- Market data from September 2025  
- Recent news articles and analyst reports on AAPL and the tech sector